In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

In [42]:
#1.1
# ============================================================
# Load dataset
# ============================================================

DATA_PATH = "FuelEconomy.csv"
df = pd.read_csv(DATA_PATH)

df = df.fillna(0)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

print("\nSummary statistics:")
display(df.describe(include="all"))

Shape: (100, 2)

Columns:
['Horse Power', 'Fuel Economy (MPG)']


,Horse Power,Fuel Economy (MPG)
0,118.770799,29.344195
1,176.326567,24.695934
2,219.262465,23.952010
3,187.310009,23.384546
4,218.594340,23.426739



Summary statistics:


,Horse Power,Fuel Economy (MPG)
count,100.000000,100.000000
mean,213.676190,23.178501
std,62.061726,4.701666
min,50.000000,10.000000
25%,174.996514,20.439516
50%,218.928402,23.143192
75%,251.706476,26.089933
max,350.000000,35.000000


In [44]:
#1.2
#Splitting data using a 70/30 split
X_train, X_test, y_train, y_test = train_test_split(df[['Horse Power']], df['Fuel Economy (MPG)'], test_size=0.3, random_state=35)

In [50]:
#1.3 and 1.4
#Defining functions to run LR and PR with varying degrees on the dataset and 1 helper module

def compute_metrics(y_true, y_pred):
    """Return MSE, MAE, R^2."""
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R^2": r2_score(y_true, y_pred),
    }
    
def run_models(X_train, X_test, y_train, y_test, scenario_name, degrees=(1, 2, 3, 4),
                            test_size=0.30, random_state=42,
                            top_k_terms=15):
    """Train/evaluate linear (deg=1) + polynomial regression models.

    Returns a DataFrame of metrics.
    """

    rows = []

    for deg in degrees:
        if deg == 1:
            model = LinearRegression()
            model_name = "Linear Regression"
        else:
            model = Pipeline([
                ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
                ("lr", LinearRegression())
            ])
            model_name = f"Polynomial Regression (degree={deg})"

        # Fit model
        model.fit(X_train, y_train)

        # Predict
        yhat_train = model.predict(X_train)
        yhat_test  = model.predict(X_test)

        # Metrics
        train_m = compute_metrics(y_train, yhat_train)
        test_m  = compute_metrics(y_test, yhat_test)

        # Report equation + plot (TEST set)
        print("\n============================================================")
        print(f"Scenario: {scenario_name}")
        print(f"Model: {model_name}")
        print("============================================================")


        rows.append({
            "Scenario": scenario_name,
            "Model": model_name,
            "Train MSE": train_m["MSE"],
            "Train MAE": train_m["MAE"],
            "Train R^2": train_m["R^2"],
            "Test MSE": test_m["MSE"],
            "Test MAE": test_m["MAE"],
            "Test R^2": test_m["R^2"],
            "Train size": len(X_train),
            "Test size": len(X_test),
        })

    return pd.DataFrame(rows)

In [52]:
#Running the fn defined above
results = run_models(X_train, X_test, y_train, y_test, scenario_name = "Regression for fuel-economy data", degrees = (1,2,3,4), test_size = 0.3, random_state=42,
                            top_k_terms=15)
print(results)


Scenario: Regression for fuel-economy data
Model: Linear Regression

Scenario: Regression for fuel-economy data
Model: Polynomial Regression (degree=2)

Scenario: Regression for fuel-economy data
Model: Polynomial Regression (degree=3)

Scenario: Regression for fuel-economy data
Model: Polynomial Regression (degree=4)
                           Scenario                             Model  \
0  Regression for fuel-economy data                 Linear Regression   
1  Regression for fuel-economy data  Polynomial Regression (degree=2)   
2  Regression for fuel-economy data  Polynomial Regression (degree=3)   
3  Regression for fuel-economy data  Polynomial Regression (degree=4)   

   Train MSE  Train MAE  Train R^2  Test MSE  Test MAE  Test R^2  Train size  \
0   2.073119   1.200258   0.888481  1.913125  1.129169  0.932885          70   
1   2.068923   1.198182   0.888707  1.868246  1.123512  0.934460          70   
2   2.029439   1.192510   0.890831  2.153429  1.224898  0.924455         

#1.5(Discussion of above results)

1) Polynomial regression with degree = 2 performs the best across each calculated evaluation metric. This is evident from the table created above that outlines each metric for every model. PR with degree = 2 has the lowest Test MSE, lowest Test MAE, and highest Test R^2.
2) No, increasing the degree of PR does not always improve tyhe performance of a model. This is evidenced by the performance of higher degree PR models above. The degree = 2 model outperforms both degree = 3 and degree = 4 models in the experiment conducted above.
3) When a model severely underperforms, it can be due to several different reasons. The biggest of which is either underfitting or overfitting. This happens when the model either does not adequately capture the complexity of data trends or if it captures too much complexity and factors noise into the model as well. Both these scenarios produce poor test metrics. Another reason this can happen is due to poor preprocessing of data as this can introduce significant skew attributed to outliers. If a model places as much emphasis on outliers as reasonable data points, it is bound to produce bad results.
4) Of the 2 reasons listed above, it is easy to see that the 3rd and 4th degree PR model produces an overfitted curve. Much of the model's complexity is captured by degree = 2 and increasing it further only captures noise. This messes with the model's ability to make good predictions. It is difficult to see the effects of bad preprocessing in the model created above as we'd have to run parallel experiments with good and bad preprocessing to make any conclusions. However, good preprocessing is considered a standard requirement to creating reliable ML models. 

In [60]:
#2.1
# ============================================================
# Load dataset
# ============================================================

DATA_PATH = "electricity_consumption_based_weather_dataset.csv"
df = pd.read_csv(DATA_PATH)

df = df.dropna()

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

print("\nSummary statistics:")
display(df.describe(include="all"))

#df.isna().sum()

Shape: (1418, 6)

Columns:
['date', 'AWND', 'PRCP', 'TMAX', 'TMIN', 'daily_consumption']


,date,AWND,PRCP,TMAX,TMIN,daily_consumption
0,2006-12-16,2.5,0.0,10.6,5.0,1209.176
1,2006-12-17,2.6,0.0,13.3,5.6,3390.460
2,2006-12-18,2.4,0.0,15.0,6.7,2203.826
3,2006-12-19,2.4,0.0,7.2,2.2,1666.194
4,2006-12-20,2.4,0.0,7.2,1.1,2225.748



Summary statistics:


,date,AWND,PRCP,TMAX,TMIN,daily_consumption
count,1418,1418.000000,1418.000000,1418.000000,1418.000000,1418.000000
unique,1418,NaN,NaN,NaN,NaN,NaN
top,2006-12-16,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.642313,3.734274,17.132722,9.085825,1562.330257
std,NaN,1.140021,10.864197,10.155059,9.044534,608.175083
min,NaN,0.000000,0.000000,-8.900000,-14.400000,14.218000
25%,NaN,1.800000,0.000000,8.900000,2.200000,1164.759500
50%,NaN,2.400000,0.000000,17.800000,9.400000,1544.028000
75%,NaN,3.300000,1.300000,26.100000,17.200000,1894.467500


date                 0
AWND                 0
PRCP                 0
TMAX                 0
TMIN                 0
daily_consumption    0
dtype: int64

In [64]:
#2.2
#Splitting data using a 70/30 split
X_train, X_test, y_train, y_test = train_test_split(df[['AWND','PRCP','TMAX','TMIN']], 
                                                    df['daily_consumption'], test_size=0.3, random_state=35)

In [66]:
#2.3,2.4
#Running the fns defined above for part 1.
results = run_models(X_train, X_test, y_train, y_test, scenario_name = "Regression for electricity based on weather", degrees = (1,2,3,4), test_size = 0.3, random_state=42,
                            top_k_terms=15)
print(results)


Scenario: Regression for fuel-economy data
Model: Linear Regression

Scenario: Regression for fuel-economy data
Model: Polynomial Regression (degree=2)

Scenario: Regression for fuel-economy data
Model: Polynomial Regression (degree=3)

Scenario: Regression for fuel-economy data
Model: Polynomial Regression (degree=4)
                           Scenario                             Model  \
0  Regression for fuel-economy data                 Linear Regression   
1  Regression for fuel-economy data  Polynomial Regression (degree=2)   
2  Regression for fuel-economy data  Polynomial Regression (degree=3)   
3  Regression for fuel-economy data  Polynomial Regression (degree=4)   

       Train MSE   Train MAE  Train R^2       Test MSE    Test MAE  Test R^2  \
0  263644.161585  380.227970   0.285001  269812.236713  387.110702  0.270871   
1  254234.469193  373.546278   0.310520  274360.438297  392.411558  0.258580   
2  248032.908666  368.317352   0.327338  271228.514422  391.759933  0.267

#2.5

1) Based on these results, the linear regression model is the best fit. However, the terrible Test R^2 metric informs us that this isn't a good prediction model. One of the reasons this could happen is that weather data is not a good indicator of electricity usage. Another reason could be the the lack of regularization and the correlation between features like Tmax and Tmin. 
2) PR models do not improve performance in this context. However, we cannot comment on whether this is due to the absence of a non-linear relationship between the 2 or the problems described above.
3) Test metrics do show the linear model outperforms the PR model. However, even then the linear model is still terrible. This can be proved using the fact that the linear model has the lowest Test MSE, lowest Test MAE, and highest Test R^2.
4) One of the reasons this could happen is that weather data is not a good indicator of electricity usage. Another reason could be the the lack of regularization and the correlation between features like Tmax and Tmin. 